In [1]:
import pandas as pd

csv_path = "../data/ML-EdgeIIoT-dataset.csv" 
df = pd.read_csv(csv_path)

print("Shape:", df.shape)

print("\nAttack_label values:")
print(df["Attack_label"].value_counts(dropna=False))

print("\nAttack_type values:")
print(df["Attack_type"].value_counts(dropna=False).head(20))

print("\nUnique source hosts:", df["ip.src_host"].nunique())
print("Unique destination hosts:", df["ip.dst_host"].nunique())

print("\nRows and label counts by source host:")
print(
    df.groupby("ip.src_host")["Attack_label"]
      .agg(rows="size", label_values="nunique")
      .sort_values("rows", ascending=False)
      .head(20)
)

Shape: (157800, 63)

Attack_label values:
Attack_label
1    133499
0     24301
Name: count, dtype: int64

Attack_type values:
Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64

Unique source hosts: 19090
Unique destination hosts: 8084

Rows and label counts by source host:
                  rows  label_values
ip.src_host                         
192.168.0.128    72546             2
192.168.0.170    47688             1
192.168.0.101    10126             2
0                 7991             2
0.0.0.0            208             2
192.168.0.152 

C:\Users\aayus\AppData\Local\Temp\ipykernel_19812\2045733828.py:4: DtypeWarning: Columns (3: arp.dst.proto_ipv4, 6: arp.src.proto_ipv4, 11: http.file_data, 13: http.request.uri.query, 14: http.request.method, 15: http.referer, 16: http.request.full_uri, 17: http.request.version, 31: tcp.options, 32: tcp.payload, 34: tcp.srcport, 39: dns.qry.name.len, 45: mqtt.conack.flags, 51: mqtt.msg, 54: mqtt.protoname, 55: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [2]:
import pandas as pd

# Show the most common source hosts and their label counts
print("Top source hosts:")
print(
    df.groupby("ip.src_host")["Attack_label"]
      .agg(rows="size", normal_rows=lambda s: (s == 0).sum(),
           attack_rows=lambda s: (s == 1).sum())
      .sort_values("rows", ascending=False)
      .head(10)
)

# Show the most common destination hosts and their label counts
print("\nTop destination hosts:")
print(
    df.groupby("ip.dst_host")["Attack_label"]
      .agg(rows="size", normal_rows=lambda s: (s == 0).sum(),
           attack_rows=lambda s: (s == 1).sum())
      .sort_values("rows", ascending=False)
      .head(10)
)

Top source hosts:
                  rows  normal_rows  attack_rows
ip.src_host                                     
192.168.0.128    72546        10125        62421
192.168.0.170    47688            0        47688
192.168.0.101    10126         9971          155
0                 7991         4190         3801
0.0.0.0            208            4          204
192.168.0.152      120            0          120
172.217.19.42       17            0           17
192.168.0.1          9            7            2
104.16.87.20         7            0            7
142.250.200.205      5            0            5

Top destination hosts:
                rows  normal_rows  attack_rows
ip.dst_host                                   
192.168.0.128  75373         9981        65392
192.168.0.170  42290            0        42290
0              20214         4190        16024
192.168.0.101  10097        10089            8
0.0             1214            0         1214
224.0.0.251      189           13        

In [3]:
# Check whether exact duplicate rows exist
print("Exact duplicate rows:", df.duplicated().sum())

# Confirm how Attack_type maps to the binary label
print("\nAttack type vs. binary label:")
print(pd.crosstab(df["Attack_type"], df["Attack_label"], margins=True))

Exact duplicate rows: 814

Attack type vs. binary label:
Attack_label               0       1     All
Attack_type                                 
Backdoor                   0   10195   10195
DDoS_HTTP                  0   10561   10561
DDoS_ICMP                  0   14090   14090
DDoS_TCP                   0   10247   10247
DDoS_UDP                   0   14498   14498
Fingerprinting             0    1001    1001
MITM                       0    1214    1214
Normal                 24301       0   24301
Password                   0    9989    9989
Port_Scanning              0   10071   10071
Ransomware                 0   10925   10925
SQL_injection              0   10311   10311
Uploading                  0   10269   10269
Vulnerability_scanner      0   10076   10076
XSS                        0   10052   10052
All                    24301  133499  157800


In [4]:
# Count missing values in each column
missing = df.isna().sum().sort_values(ascending=False)

print("Columns with missing values:")
print(missing[missing > 0])

# Show each column's pandas data type
print("\nColumn data types:")
print(df.dtypes.to_string())

# Check whether numeric columns contain infinity values
import numpy as np

numeric_df = df.select_dtypes(include="number")
infinity_counts = np.isinf(numeric_df).sum()

print("\nNumeric columns with infinity values:")
print(infinity_counts[infinity_counts > 0])

Columns with missing values:
Series([], dtype: int64)

Column data types:
frame.time                       str
ip.src_host                      str
ip.dst_host                      str
arp.dst.proto_ipv4            object
arp.opcode                   float64
arp.hw.size                  float64
arp.src.proto_ipv4            object
icmp.checksum                float64
icmp.seq_le                  float64
icmp.transmit_timestamp      float64
icmp.unused                  float64
http.file_data                object
http.content_length          float64
http.request.uri.query        object
http.request.method           object
http.referer                  object
http.request.full_uri         object
http.request.version          object
http.response                float64
http.tls_port                float64
tcp.ack                      float64
tcp.ack_raw                  float64
tcp.checksum                 float64
tcp.connection.fin           float64
tcp.connection.rst           float64
t

In [5]:
import numpy as np
import pandas as pd

# Keep numeric columns only, excluding the target columns
excluded = {"Attack_label", "Attack_type"}
numeric_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col not in excluded
]

X_numeric = df[numeric_cols].copy()
y = df["Attack_label"].astype("int64").copy()

# Find constant columns (same value in every row)
constant_cols = X_numeric.columns[X_numeric.nunique(dropna=False) <= 1].tolist()

print("Numeric feature count:", len(numeric_cols))
print("Constant numeric columns:", constant_cols)
print("Feature matrix shape:", X_numeric.shape)
print("Target shape:", y.shape)

print("\nClass counts:")
print(y.value_counts())

Numeric feature count: 42
Constant numeric columns: ['icmp.unused', 'http.tls_port', 'dns.qry.type', 'dns.retransmit_request_in', 'mqtt.msg_decoded_as', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']
Feature matrix shape: (157800, 42)
Target shape: (157800,)

Class counts:
Attack_label
1    133499
0     24301
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import GroupShuffleSplit

# Remove constant columns
constant_cols = X_numeric.columns[X_numeric.nunique(dropna=False) <= 1].tolist()
X = X_numeric.drop(columns=constant_cols).copy()

# Group exact duplicate rows together, using all original columns
groups = pd.util.hash_pandas_object(df, index=False)

# First split: 70% train, 30% temporary
split1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(split1.split(X, y, groups=groups))

# Second split: divide temporary data equally into validation and test
split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel_idx, test_rel_idx = next(
    split2.split(X.iloc[temp_idx], y.iloc[temp_idx], groups=groups.iloc[temp_idx])
)

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

# Create the splits
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

print("Split sizes:")
print("Train:", X_train.shape, f"({len(X_train) / len(X):.1%})")
print("Validation:", X_val.shape, f"({len(X_val) / len(X):.1%})")
print("Test:", X_test.shape, f"({len(X_test) / len(X):.1%})")

print("\nAttack-label proportions:")
for name, labels in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test),
]:
    print(f"\n{name}:")
    print(labels.value_counts(normalize=True).sort_index().rename({0: "Normal", 1: "Attack"}))

# Verify that no exact duplicate group crosses splits
train_groups = set(groups.iloc[train_idx])
val_groups = set(groups.iloc[val_idx])
test_groups = set(groups.iloc[test_idx])

print("\nDuplicate groups shared across splits:")
print("Train/validation:", len(train_groups & val_groups))
print("Train/test:", len(train_groups & test_groups))
print("Validation/test:", len(val_groups & test_groups))

Split sizes:
Train: (110071, 34) (69.8%)
Validation: (23548, 34) (14.9%)
Test: (24181, 34) (15.3%)

Attack-label proportions:

Train:
Attack_label
Normal    0.154782
Attack    0.845218
Name: proportion, dtype: float64

Validation:
Attack_label
Normal    0.154663
Attack    0.845337
Name: proportion, dtype: float64

Test:
Attack_label
Normal    0.149787
Attack    0.850213
Name: proportion, dtype: float64

Duplicate groups shared across splits:
Train/validation: 0
Train/test: 0
Validation/test: 0


In [7]:
import numpy as np

# Dirichlet alpha controls how different client label distributions are.
# Smaller alpha -> stronger label imbalance between clients.
alpha = 5
num_clients = 7
rng = np.random.default_rng(42)

y_train_array = y_train.to_numpy()
client_positions = [[] for _ in range(num_clients)]

# Distribute each class across clients independently
for label in np.unique(y_train_array):
    label_positions = np.where(y_train_array == label)[0]
    rng.shuffle(label_positions)

    proportions = rng.dirichlet(np.full(num_clients, alpha))
    counts = rng.multinomial(len(label_positions), proportions)

    start = 0
    for client_id, count in enumerate(counts):
        client_positions[client_id].extend(
            label_positions[start:start + count]
        )
        start += count

# Shuffle each client's assigned positions and create client datasets
clients = {}

for client_id, positions in enumerate(client_positions):
    positions = np.array(positions)
    rng.shuffle(positions)

    clients[client_id] = {
        "X_train": X_train.iloc[positions].copy(),
        "y_train": y_train.iloc[positions].copy(),
    }

# Check each client's size and label distribution
print("Client data summary:")
for client_id, client_data in clients.items():
    labels = client_data["y_train"]
    counts = labels.value_counts().to_dict()
    attack_rate = (labels == 1).mean()

    print(
        f"Client {client_id}: "
        f"{len(labels):,} rows | "
        f"normal={counts.get(0, 0):,} | "
        f"attack={counts.get(1, 0):,} | "
        f"attack rate={attack_rate:.1%}"
    )

print("\nTotal client training rows:",
      sum(len(client_data["y_train"]) for client_data in clients.values()))
print("Original training rows:", len(X_train))

Client data summary:
Client 0: 29,073 rows | normal=3,769 | attack=25,304 | attack rate=87.0%
Client 1: 10,395 rows | normal=3,146 | attack=7,249 | attack rate=69.7%
Client 2: 15,389 rows | normal=2,945 | attack=12,444 | attack rate=80.9%
Client 3: 14,892 rows | normal=1,170 | attack=13,722 | attack rate=92.1%
Client 4: 9,085 rows | normal=1,464 | attack=7,621 | attack rate=83.9%
Client 5: 16,947 rows | normal=2,813 | attack=14,134 | attack rate=83.4%
Client 6: 14,290 rows | normal=1,730 | attack=12,560 | attack rate=87.9%

Total client training rows: 110071
Original training rows: 110071


In [8]:
from sklearn.preprocessing import StandardScaler

# Fit only on the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Replace each client's feature DataFrame with its scaled NumPy array
for client_id in clients:
    clients[client_id]["X_train_scaled"] = scaler.transform(
        clients[client_id]["X_train"]
    )

print("Scaled train:", X_train_scaled.shape)
print("Scaled validation:", X_val_scaled.shape)
print("Scaled test:", X_test_scaled.shape)

print("\nClient 0 scaled shape:", clients[0]["X_train_scaled"].shape)

Scaled train: (110071, 34)
Scaled validation: (23548, 34)
Scaled test: (24181, 34)

Client 0 scaled shape: (29073, 34)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

local_models = {}
results = []

for client_id, client_data in clients.items():
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(
        client_data["X_train_scaled"],
        client_data["y_train"]
    )

    y_pred = model.predict(X_test_scaled)

    local_models[client_id] = model
    results.append({
        "client": client_id,
        "train_rows": len(client_data["y_train"]),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results).set_index("client")
print(results_df.round(4))

        train_rows  accuracy  precision  recall      f1
client                                                 
0            29073    0.7167     0.9773  0.6827  0.8038
1            10395    0.7163     0.9730  0.6854  0.8043
2            15389    0.7213     0.9718  0.6923  0.8085
3            14892    0.7164     0.9746  0.6843  0.8041
4             9085    0.7228     0.9688  0.6964  0.8103
5            16947    0.7261     0.9672  0.7017  0.8133
6            14290    0.7154     0.9726  0.6845  0.8035


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

central_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

central_model.fit(X_train_scaled, y_train)
central_pred = central_model.predict(X_test_scaled)

print("Centralized Logistic Regression")
print("Accuracy :", round(accuracy_score(y_test, central_pred), 4))
print("Precision:", round(precision_score(y_test, central_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, central_pred, zero_division=0), 4))
print("F1       :", round(f1_score(y_test, central_pred, zero_division=0), 4))
print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, central_pred))

Centralized Logistic Regression
Accuracy : 0.7165
Precision: 0.9772
Recall   : 0.6825
F1       : 0.8037

Confusion matrix [[TN, FP], [FN, TP]]:
[[ 3294   328]
 [ 6528 14031]]


In [11]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.15.0+cu132
CUDA available: True


In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use CUDA when available; otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

input_dim = X_train_scaled.shape[1]

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

# Create the model and move it to the GPU
model = BinaryClassifier(input_dim).to(device)

# Convert validation and test data to tensors on the selected device
X_val_tensor = torch.tensor(
    X_val_scaled, dtype=torch.float32, device=device
)
y_val_tensor = torch.tensor(
    y_val.to_numpy(), dtype=torch.float32, device=device
)

X_test_tensor = torch.tensor(
    X_test_scaled, dtype=torch.float32, device=device
)
y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32, device=device
)

print("Input features:", input_dim)
print("Train rows:", len(X_train_scaled))
print("Validation rows:", len(X_val_tensor))
print("Test rows:", len(X_test_tensor))
print("Model device:", next(model.parameters()).device)

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Input features: 34
Train rows: 110071
Validation rows: 23548
Test rows: 24181
Model device: cuda:0


In [13]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=256,
    shuffle=True
)

# Example batch loop pattern
for X_batch, y_batch in train_loader:
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    # Forward pass and training steps go here

In [16]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# -------------------------
# Settings
# -------------------------
num_rounds = 10
local_epochs = 1
batch_size = 256
learning_rate = 0.001
threshold = 0.5

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

# -------------------------
# Convert evaluation data
# -------------------------
X_val_tensor = torch.as_tensor(
    X_val_scaled, dtype=torch.float32, device=device
)
y_val_array = y_val.to_numpy().astype(np.float32)

X_test_tensor = torch.as_tensor(
    X_test_scaled, dtype=torch.float32, device=device
)
y_test_array = y_test.to_numpy().astype(np.int64)

# -------------------------
# Loss and global model
# -------------------------
# Use the global training label ratio to weight the positive class
normal_count = int((y_train == 0).sum())
attack_count = int((y_train == 1).sum())

pos_weight = torch.tensor(
    [normal_count / attack_count],
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

global_model = BinaryClassifier(input_dim).to(device)

# Keep the global weights on CPU between rounds
global_state = {
    key: value.detach().cpu().clone()
    for key, value in global_model.state_dict().items()
}

# -------------------------
# Federated training
# -------------------------
for round_num in range(1, num_rounds + 1):
    client_states = []
    client_sizes = []

    for client_id, client_data in clients.items():
        # Use the scaled training data prepared for this client
        X_client = torch.as_tensor(
            client_data["X_train_scaled"], dtype=torch.float32
        )
        y_client = torch.tensor(
            client_data["y_train"].to_numpy(copy=True),
            dtype=torch.float32
        )

        client_loader = DataLoader(
            TensorDataset(X_client, y_client),
            batch_size=batch_size,
            shuffle=True,
        )

        # Start each client from the same current global model
        local_model = BinaryClassifier(input_dim).to(device)
        local_model.load_state_dict(global_state)
        local_model.train()

        optimizer = torch.optim.Adam(
            local_model.parameters(),
            lr=learning_rate,
        )

        for _ in range(local_epochs):
            for X_batch, y_batch in client_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                optimizer.zero_grad()
                logits = local_model(X_batch)
                loss = criterion(logits, y_batch)
                loss.backward()
                optimizer.step()

        # Save updated client weights on CPU
        client_states.append({
            key: value.detach().cpu().clone()
            for key, value in local_model.state_dict().items()
        })
        client_sizes.append(len(y_client))

        del local_model, optimizer, client_loader, X_client, y_client

    # FedAvg: average client weights by their training-set sizes
    total_client_rows = sum(client_sizes)
    new_global_state = {}

    for key in global_state:
        new_global_state[key] = sum(
            client_states[i][key] * (client_sizes[i] / total_client_rows)
            for i in range(len(client_states))
        )

    global_state = new_global_state
    global_model.load_state_dict(global_state)

    # Validate after each round; do not use test data for model selection
    global_model.eval()
    with torch.no_grad():
        val_logits = global_model(X_val_tensor)
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_preds = (val_probs >= threshold).astype(np.int64)

    print(
        f"Round {round_num}/{num_rounds} | "
        f"Val F1: {f1_score(y_val, val_preds, zero_division=0):.4f} | "
        f"Val Precision: {precision_score(y_val, val_preds, zero_division=0):.4f} | "
        f"Val Recall: {recall_score(y_val, val_preds, zero_division=0):.4f}"
    )

# -------------------------
# Final test evaluation
# -------------------------
global_model.eval()

with torch.no_grad():
    test_logits = global_model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits).cpu().numpy()
    test_preds = (test_probs >= threshold).astype(np.int64)

print("\nFinal FedAvg test results")
print("Accuracy :", round(accuracy_score(y_test_array, test_preds), 4))
print("Precision:", round(precision_score(y_test_array, test_preds, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test_array, test_preds, zero_division=0), 4))
print("F1       :", round(f1_score(y_test_array, test_preds, zero_division=0), 4))
print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test_array, test_preds))

print("\nFinal global model device:", next(global_model.parameters()).device)

Training device: cuda
Round 1/10 | Val F1: 0.7548 | Val Precision: 0.9467 | Val Recall: 0.6275
Round 2/10 | Val F1: 0.8630 | Val Precision: 0.9339 | Val Recall: 0.8022
Round 3/10 | Val F1: 0.8631 | Val Precision: 0.9341 | Val Recall: 0.8022
Round 4/10 | Val F1: 0.8372 | Val Precision: 0.9799 | Val Recall: 0.7308
Round 5/10 | Val F1: 0.8582 | Val Precision: 0.9775 | Val Recall: 0.7648
Round 6/10 | Val F1: 0.8649 | Val Precision: 0.9798 | Val Recall: 0.7740
Round 7/10 | Val F1: 0.8885 | Val Precision: 0.9751 | Val Recall: 0.8160
Round 8/10 | Val F1: 0.8904 | Val Precision: 0.9782 | Val Recall: 0.8170
Round 9/10 | Val F1: 0.8917 | Val Precision: 0.9774 | Val Recall: 0.8199
Round 10/10 | Val F1: 0.8931 | Val Precision: 0.9800 | Val Recall: 0.8204

Final FedAvg test results
Accuracy : 0.8367
Precision: 0.9812
Recall   : 0.8238
F1       : 0.8956

Confusion matrix [[TN, FP], [FN, TP]]:
[[ 3297   325]
 [ 3623 16936]]

Final global model device: cuda:0


In [17]:
# FedAvg training with best-validation checkpointing
# Assumes these already exist:
# global_model, client_data, X_val_scaled, y_val, X_test_scaled, y_test

from pathlib import Path
import copy
import json

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# --------------------
# Settings
# --------------------
NUM_ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 256
LEARNING_RATE = 0.001
THRESHOLD = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
global_model = global_model.to(device)

# --------------------
# Helpers
# --------------------
def features_to_tensor(values):
    return torch.tensor(
        np.asarray(values).copy(),
        dtype=torch.float32,
    )

def labels_to_tensor(values):
    return torch.tensor(
        np.asarray(values).reshape(-1, 1).copy(),
        dtype=torch.float32,
    )

# Validation and test tensors
X_val_tensor = features_to_tensor(X_val_scaled).to(device)
y_val_tensor = labels_to_tensor(y_val).to(device)

X_test_tensor = features_to_tensor(X_test_scaled).to(device)
y_test_tensor = labels_to_tensor(y_test).to(device)

# Calculate positive-class weight from all client training labels.
# This preserves the weighting approach used in the earlier FedAvg run.
all_train_labels = np.concatenate([
    np.asarray(client["y_train"]).reshape(-1)
    for client in client_data.values()
])

num_positive = int((all_train_labels == 1).sum())
num_negative = int((all_train_labels == 0).sum())

if num_positive == 0 or num_negative == 0:
    raise ValueError("Training data must contain both label classes: 0 and 1.")

pos_weight = torch.tensor(
    [num_negative / num_positive],
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# --------------------
# Best-checkpoint tracking
# --------------------
best_val_f1 = -1.0
best_round = None
best_state = None

# --------------------
# FedAvg training rounds
# --------------------
for round_num in range(1, NUM_ROUNDS + 1):
    client_states = []
    client_sizes = []

    # Every client starts from the current global model.
    for client_id, client in client_data.items():
        X_client = features_to_tensor(client["X_train_scaled"])
        y_client = labels_to_tensor(client["y_train"])

        train_loader = DataLoader(
            TensorDataset(X_client, y_client),
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

        local_model = copy.deepcopy(global_model).to(device)
        local_model.train()

        optimizer = torch.optim.Adam(
            local_model.parameters(),
            lr=LEARNING_RATE,
        )

        for _ in range(LOCAL_EPOCHS):
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()
                logits = local_model(batch_X)
                loss = criterion(logits, batch_y)
                loss.backward()
                optimizer.step()

        # Keep local weights on CPU for aggregation.
        client_states.append({
            name: tensor.detach().cpu().clone()
            for name, tensor in local_model.state_dict().items()
        })
        client_sizes.append(len(X_client))

        del local_model

    # Weighted FedAvg: each client is weighted by its number of training rows.
    total_rows = sum(client_sizes)
    aggregated_state = {}

    for name in client_states[0]:
        aggregated_state[name] = sum(
            client_states[i][name] * (client_sizes[i] / total_rows)
            for i in range(len(client_states))
        )

    global_model.load_state_dict(aggregated_state)
    global_model.to(device)

    # Validation evaluation: used only to choose the checkpoint.
    global_model.eval()

    with torch.no_grad():
        val_logits = global_model(X_val_tensor)
        val_probs = torch.sigmoid(val_logits)
        val_preds = (val_probs >= THRESHOLD).int().cpu().numpy().reshape(-1)
        val_true = y_val_tensor.int().cpu().numpy().reshape(-1)

    val_f1 = f1_score(val_true, val_preds, zero_division=0)
    val_precision = precision_score(val_true, val_preds, zero_division=0)
    val_recall = recall_score(val_true, val_preds, zero_division=0)

    print(
        f"Round {round_num}/{NUM_ROUNDS} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val precision: {val_precision:.4f} | "
        f"Val recall: {val_recall:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_round = round_num
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in global_model.state_dict().items()
        }

# --------------------
# Restore best validation checkpoint
# --------------------
if best_state is None:
    raise RuntimeError("No validation checkpoint was saved.")

global_model.load_state_dict(best_state)
global_model.to(device)
global_model.eval()

print(f"\nSelected round: {best_round}")
print(f"Best validation F1: {best_val_f1:.4f}")

# --------------------
# Test evaluation: run once, after checkpoint selection
# --------------------
with torch.no_grad():
    test_logits = global_model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs >= THRESHOLD).int().cpu().numpy().reshape(-1)
    test_true = y_test_tensor.int().cpu().numpy().reshape(-1)

test_metrics = {
    "selected_round": int(best_round),
    "best_validation_f1": float(best_val_f1),
    "test_accuracy": float(accuracy_score(test_true, test_preds)),
    "test_precision": float(precision_score(test_true, test_preds, zero_division=0)),
    "test_recall": float(recall_score(test_true, test_preds, zero_division=0)),
    "test_f1": float(f1_score(test_true, test_preds, zero_division=0)),
    "threshold": THRESHOLD,
    "num_rounds": NUM_ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "pos_weight": float(pos_weight.item()),
    "confusion_matrix": confusion_matrix(test_true, test_preds).tolist(),
}

print("\nTest metrics for the selected checkpoint:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value}")

# --------------------
# Save checkpoint and metrics
# --------------------
# This path assumes the notebook is inside the project's notebooks/ folder.
output_dir = Path("../outputs/edgeiiot_fedavg")
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "best_model.pt"
metrics_path = output_dir / "metrics.json"

torch.save(
    {
        "round": best_round,
        "best_val_f1": best_val_f1,
        "model_state_dict": best_state,
    },
    checkpoint_path,
)

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=2)

print(f"\nCheckpoint saved to: {checkpoint_path}")
print(f"Metrics saved to:    {metrics_path}")

KeyError: 'y_train'